# 06 — One-click Colab submission runner

This notebook is designed to work from a **blank Google Colab GPU runtime**. It clones or refreshes the canonical repository first, installs dependencies, validates CUDA, then executes the complete graded pipeline.

**Required:** Runtime → Change runtime type → GPU. Keep the same GPU runtime for all required experiments so throughput and memory remain comparable.

In [ ]:
from pathlib import Path
import os, subprocess

REPO_URL = "https://github.com/JoeIndyGit/era-v5-session-13-reversible-llm-lab.git"
REPO_DIR = Path("/content/era-v5-session-13-reversible-llm-lab")

if not REPO_DIR.exists():
    subprocess.run(["git", "clone", REPO_URL, str(REPO_DIR)], check=True)
else:
    subprocess.run(["git", "-C", str(REPO_DIR), "pull", "--ff-only"], check=True)

os.chdir(REPO_DIR)
print("Repository:", REPO_DIR)
print(subprocess.check_output(["git", "rev-parse", "HEAD"], text=True).strip())

In [ ]:
!pip -q install -r requirements.txt

In [ ]:
import torch, subprocess
assert torch.cuda.is_available(), "GPU is not enabled. Select Runtime → Change runtime type → GPU."
gpu = torch.cuda.get_device_name(0)
mem = torch.cuda.get_device_properties(0).total_memory / 1024**3
print(f"CUDA ready: {gpu} | {mem:.2f} GiB")
subprocess.run(["nvidia-smi"], check=True)

## Execute the full graded protocol

The runner performs correctness validation, dataset/tokenizer preparation, Midpoint-vs-Leapfrog selection, all three required 50M-token runs, both 10-update batch-frontier searches, the final evidence audit, and report generation. Existing authoritative result files are skipped after a runtime restart unless `--force` is used.

In [ ]:
!python scripts/run_full_submission.py

## Package the evidence

This creates a compact artifact containing the README, raw JSON/CSV evidence, environment snapshot, generated figures, configs, source code, scripts, and notebooks. The large TinyStories cache is intentionally excluded.

In [ ]:
!python scripts/package_evidence.py

In [ ]:
from pathlib import Path
import json
print(Path("submission_evidence/MANIFEST.json").read_text())

## Completion gate

The experiment is submission-complete only when the console contains:

```text
FINAL EVIDENCE AUDIT: PASS
COMPLETE: required evidence is in results/, figures in assets/, README is populated.
```

and `submission_evidence/era-v5-session-13-evidence.zip` has been generated. Commit the measured `results/`, generated `assets/`, updated `README.md`, and executed notebooks to GitHub.